# Data Quality — tokopedia

Jalankan **Run All** dengan kernel `env`. Keenam pemeriksaan di bawah memakai aturan yang sama untuk semua sumber. Hasil transaksi mengikuti 12 kolom tanpa customer_id; `product_id` memakai SKU asli dari Product Master.

Product Master tetap berisi identitas produk dan harga referensi. Data sumber tetap utuh. Semua aturan dijalankan saat persiapan agar pemeriksaan duplikat sudah memperhitungkan hasil mapping produk; enam bagian berikut memperlihatkan hasil tiap pemeriksaan.

In [9]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pipeline/validation/analysis.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.validation.analysis import (
    load_analysis, standard_data, missing_values, quality_issues,
    type_report, summary, export_analysis,
)

SOURCE = "tokopedia"
hasil = load_analysis(SOURCE, ROOT / "data/source")
data_bersih = standard_data(hasil)


## 1. Missing value

Field wajib: order ID, tanggal, produk, quantity, harga satuan, status; total transaksi Website juga wajib. Semua field master wajib. Baris yang tidak memenuhi syarat ditolak. Customer/kota/email yang tidak tersedia tetap kosong, tanpa dummy. Field opsional kosong yang tersedia dalam source dicatat sebagai warning.

In [10]:
display(missing_values(hasil))
display(quality_issues(hasil, "missing"))

,sumber,kolom,jumlah_kosong
0,tokopedia,transaction_id,0
1,tokopedia,transaction_date,0
2,tokopedia,item_name,0
3,tokopedia,quantity,0
4,tokopedia,price,0
5,tokopedia,buyer_name,4
6,tokopedia,city,6
7,tokopedia,payment,6
8,tokopedia,status,0


,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,28,WARNING,city,MISSING_OPTIONAL,
1,tokopedia,30,WARNING,buyer_name,MISSING_OPTIONAL,
2,tokopedia,35,WARNING,payment,MISSING_OPTIONAL,
3,tokopedia,53,WARNING,buyer_name,MISSING_OPTIONAL,
4,tokopedia,179,WARNING,city,MISSING_OPTIONAL,
5,tokopedia,180,WARNING,payment,MISSING_OPTIONAL,
6,tokopedia,199,WARNING,payment,MISSING_OPTIONAL,
7,tokopedia,203,WARNING,city,MISSING_OPTIONAL,
8,tokopedia,209,WARNING,city,MISSING_OPTIONAL,
9,tokopedia,210,WARNING,buyer_name,MISSING_OPTIONAL,


## 2. Duplicate

Business key: `(channel, order_id)` untuk dataset satu item per order saat ini; master memakai `product_id`/SKU. Record yang sama disimpan satu kali. Jika key sama tetapi nilainya berbeda, semua versi ditolak untuk ditinjau. Bila kelak order memiliki beberapa item, tambahkan line ID dari sumber.

In [11]:
display(quality_issues(hasil, "duplicate"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,136,INFO,business_key,DUPLICATE_BUSINESS_KEY,TKP-0106


## 3. Invalid value

Quantity wajib integer positif. Harga wajib positif dan maksimal dua desimal. Total harus sama dengan quantity × harga satuan. Status dataset saat ini: `Completed`, `Cancelled`, `Returned`; status lain ditolak. Harga berbeda dari master diberi warning karena mungkin promo. Total harga adalah nilai bruto; hitung penjualan selesai hanya dari `Completed`.

In [12]:
display(quality_issues(hasil, "invalid"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,2,ERROR,quantity,NON_POSITIVE,0
1,tokopedia,2,WARNING,price,PRICE_DIFFERS_FROM_MASTER,27900
2,tokopedia,3,WARNING,price,PRICE_DIFFERS_FROM_MASTER,27900
3,tokopedia,4,WARNING,price,PRICE_DIFFERS_FROM_MASTER,45900
4,tokopedia,7,ERROR,price,NON_POSITIVE,-50000
...,...,...,...,...,...,...
173,tokopedia,384,WARNING,price,PRICE_DIFFERS_FROM_MASTER,43900
174,tokopedia,389,ERROR,price,NON_POSITIVE,0
175,tokopedia,391,WARNING,price,PRICE_DIFFERS_FROM_MASTER,218000
176,tokopedia,394,WARNING,price,PRICE_DIFFERS_FROM_MASTER,43900


## 4. Date format

Tanggal Shopee/Tokopedia: DD/MM/YYYY; Website: MMM DD, YYYY; Offline: DD-MMM-YYYY. Hasil CSV selalu YYYY-MM-DD. Tanggal tidak valid ditolak. Product Master tidak memiliki tanggal transaksi.

In [13]:
display(quality_issues(hasil, "date"))
if "tanggal_order" in data_bersih:
    display(data_bersih[["order_id", "tanggal_order"]].head(5))
else:
    print("Tidak berlaku: master produk tidak memiliki tanggal transaksi.")

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,121,ERROR,transaction_date,INVALID_DATE,not-a-date
1,tokopedia,143,ERROR,transaction_date,INVALID_DATE,not-a-date
2,tokopedia,152,ERROR,transaction_date,INVALID_DATE,not-a-date
3,tokopedia,232,ERROR,transaction_date,INVALID_DATE,not-a-date
4,tokopedia,284,ERROR,transaction_date,INVALID_DATE,not-a-date
5,tokopedia,289,ERROR,transaction_date,INVALID_DATE,not-a-date
6,tokopedia,332,ERROR,transaction_date,INVALID_DATE,not-a-date
7,tokopedia,346,ERROR,transaction_date,INVALID_DATE,not-a-date
8,tokopedia,373,ERROR,transaction_date,INVALID_DATE,not-a-date


,order_id,tanggal_order
0,TKP-0244,2026-07-02
1,TKP-0419,2026-05-09
2,TKP-0317,2026-05-02
3,TKP-0318,2026-08-20
4,TKP-0489,2026-07-26


## 5. Data type

Identifier string, quantity Int64, tanggal datetime, dan uang Decimal. CSV tidak menyimpan tipe data; ekspor tanggal menggunakan YYYY-MM-DD. Teks dirapikan spasinya tanpa merusak nama brand, SKU, shade, atau SPF/PA++++.

In [14]:
display(type_report(data_bersih))

,kolom,dtype,tipe_nilai
0,order_id,string,str
1,product_id,string,str
2,product_name,string,str
3,kategori,string,str
4,quantity,Int64,int64
5,total_harga,object,Decimal
6,tanggal_order,datetime64[us],Timestamp
7,kota,string,str
8,channel,string,str
9,status,string,str


## 6. Product consistency

Variasi huruf besar/kecil, spasi, underscore, dan hyphen dicocokkan ke master. Nama produk dan kategori mengikuti master. Produk tidak dikenal/typo ambigu ditolak, tanpa menebak SKU. Pada master, SKU harus unik.

In [15]:
display(data_bersih[["product_id", "product_name", "kategori"]].drop_duplicates())
display(quality_issues(hasil, "product"))

,product_id,product_name,kategori
0,EMN-MUP-001,Emina Cheek Lit Cream Blush Peach,Makeup
1,WRD-SKC-002,Wardah Lightening Micellar Water 100ml,Skincare
4,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup
6,WRD-SKC-003,Wardah Aloe Hydramild Moisturizer 40ml,Skincare
7,WND-BDY-001,Wonderly Body Mist Sweet Blossom 100ml,Fragrance
8,LBR-SKC-002,LABORE Barrier Revive Cream 30ml,Skincare
9,KHF-BDY-001,Kahf Face Wash Oil and Comedo Defense 100ml,Mens Grooming
10,EMN-SKC-001,Emina Bright Stuff Face Wash 100ml,Skincare
11,WRD-MUP-002,Wardah Colorfit Perfect Glow Cushion 13N,Makeup
15,LBR-SKC-001,LABORE BiomeProtect Physical Sunscreen SPF 50 ...,Skincare


,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,5,ERROR,item_name,UNMAPPED_PRODUCT,Putri Hair Treatment Shampoo 250ml
1,tokopedia,10,ERROR,item_name,UNMAPPED_PRODUCT,Putri Hair Treatment Shampoo 250ml
2,tokopedia,17,ERROR,item_name,UNMAPPED_PRODUCT,Nourish Hand Cream 50ml
3,tokopedia,25,ERROR,item_name,UNMAPPED_PRODUCT,Brigthning Serumm 30ml
4,tokopedia,31,ERROR,item_name,UNMAPPED_PRODUCT,Nourish Hand Cream 50ml
5,tokopedia,32,ERROR,item_name,UNMAPPED_PRODUCT,Putri Hair Treatment Shampoo 250ml
6,tokopedia,35,ERROR,item_name,UNMAPPED_PRODUCT,Nourish Hand Cream 50ml
7,tokopedia,53,ERROR,item_name,UNMAPPED_PRODUCT,Glowly Vitamin C Serum 20ml
8,tokopedia,67,ERROR,item_name,UNMAPPED_PRODUCT,Putri Hair Treatment Shampoo 250ml
9,tokopedia,79,ERROR,item_name,UNMAPPED_PRODUCT,Glowly Vitamin C Serum 20ml


## Hasil akhir

Format dan urutan kolom sama untuk semua transaksi. `kota` adalah kota pelanggan online atau kota toko offline; Website yang tidak memiliki kota tetap kosong. Nama channel tetap Shopee/Tokopedia/Website/Offline Store agar bisa dibandingkan.

Hasil utama: `data/processed/clean/`. Notebook ini menyimpan file bersih sumber yang dibahas; `analisa.ipynb` menyimpan seluruh sumber, `sales.csv`, `summary.csv`, serta satu `quality_issues.csv` untuk detail masalah.

In [16]:
ringkasan = summary(hasil)
assert (ringkasan["awal"] == ringkasan["bersih"] + ringkasan["duplikat"] + ringkasan["ditolak"]).all()
display(ringkasan)
display(data_bersih.head(10))
folder_hasil = export_analysis(hasil)
print("Tersimpan:", folder_hasil)

,sumber,awal,bersih,duplikat,ditolak
0,tokopedia,400,334,1,65


,order_id,product_id,product_name,kategori,quantity,total_harga,tanggal_order,kota,channel,status,customer_email,harga_satuan,customer_id
0,TKP-0244,EMN-MUP-001,Emina Cheek Lit Cream Blush Peach,Makeup,5,234500.00,2026-07-02,Semarang,Tokopedia,Completed,<NA>,46900.00,<NA>
1,TKP-0419,WRD-SKC-002,Wardah Lightening Micellar Water 100ml,Skincare,1,27900.00,2026-05-09,Jakarta,Tokopedia,Cancelled,<NA>,27900.00,<NA>
2,TKP-0317,EMN-MUP-001,Emina Cheek Lit Cream Blush Peach,Makeup,2,91800.00,2026-05-02,Yogyakarta,Tokopedia,Cancelled,<NA>,45900.00,<NA>
3,TKP-0318,WRD-SKC-002,Wardah Lightening Micellar Water 100ml,Skincare,5,144500.00,2026-08-20,Semarang,Tokopedia,Completed,<NA>,28900.00,<NA>
4,TKP-0489,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup,2,128800.00,2026-07-26,Surabaya,Tokopedia,Completed,<NA>,64400.00,<NA>
5,TKP-0344,WRD-SKC-002,Wardah Lightening Micellar Water 100ml,Skincare,3,95700.00,2026-01-27,Jakarta,Tokopedia,Completed,<NA>,31900.00,<NA>
6,TKP-0306,WRD-SKC-003,Wardah Aloe Hydramild Moisturizer 40ml,Skincare,2,91800.00,2026-05-01,Yogyakarta,Tokopedia,Completed,<NA>,45900.00,<NA>
7,TKP-0161,WND-BDY-001,Wonderly Body Mist Sweet Blossom 100ml,Fragrance,4,183600.00,2026-01-04,Jakarta,Tokopedia,Cancelled,<NA>,45900.00,<NA>
8,TKP-0340,LBR-SKC-002,LABORE Barrier Revive Cream 30ml,Skincare,2,241000.00,2026-06-20,Yogyakarta,Tokopedia,Completed,<NA>,120500.00,<NA>
9,TKP-0180,KHF-BDY-001,Kahf Face Wash Oil and Comedo Defense 100ml,Mens Grooming,3,128700.00,2026-06-06,Yogyakarta,Tokopedia,Returned,<NA>,42900.00,<NA>


Tersimpan: C:\Users\ADVAN\OneDrive - Universitas Teknologi Yogyakarta\Rinaldi\Ecommerce Sales\data\processed\clean
